In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:98% !important;}
div.cell.code_cell.rendered{width:98%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li, div.text_cell_render p{width:95% !important;font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

## RAG 절차
- 1. docx문서를 읽는다(python-docx) 
- 2. 읽어온 문서를 쪼갠다(text_spliter이용)
    * 모델의 context window와 비용 확인
    * chunk(문서를 쪼갠 하나)가 길면 비용과 시간이 많이 소요
- 3. 쪼갠 문서를 임베딩 -> vector Store에 저장 => chroma(local 파일)
- 4. 질문과 vector DB의 유사도 검색
- 5. 유사도검색으로 가져온 문서를 LLM에 질문과 같이 전달하여 답변 생성(LangChain)

# 0. 패키지 설치

In [2]:
# 문서읽어오기
#%pip install -q docx2txt

In [11]:
# 텍스트를 chunk로 나누는 기능 
%pip install -q langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# 파일 저장 chroma (cf. chromadb와 다름)
%pip install -q langchain-chroma

In [9]:
# langchain 사용
%pip install -q langchain

Note: you may need to restart the kernel to use updated packages.


# 1. 문서읽기(x)

In [14]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
loader = Docx2txtLoader('./tax_docx/소득세법(법률)(제21221호)(20260701).docx')
document = loader.load()
len(document)

CPU times: total: 4 s
Wall time: 3.99 s


1

# 2. 문서를 쪼개면서 읽기(o)
## 2.1 1500토큰씩 겹쳐서 쪼개면서 읽기

In [30]:
import time
start = time.time()
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import TokenTextSplitter
loader = Docx2txtLoader('./tax_docx/소득세법(법률)(제21221호)(20260701).docx')
# gpt-4o-mini, .... 모델의 토큰 추출 방식 : o200k_base방식
# gpt-4,  gpt4 turbo .. 모델의 토큰 추출방식 : cl100k_base방식
text_splitter = TokenTextSplitter(
        encoding_name="cl100k_base", # 토큰을 세는 방식 이름
        chunk_size=1500,             # 1500토큰 수 기준으로 chunk 분리
        chunk_overlap=200,
        # separator 파라미터가 없어
)
documents = loader.load_and_split(text_splitter=text_splitter)
runtime = time.time() - start
print('문서를 겹쳐 쪼개면서 읽어오는 시간 :', runtime)
print('chunk갯수 :', len(documents))

문서를 겹쳐 쪼개면서 읽어오는 시간 : 4.113900184631348
chunk갯수 : 180


In [24]:
# 각 chunk별 글자수 : len(documents[0].page_content)
print([len(document.page_content) for document in documents] )

[1718, 1657, 1639, 1649, 1738, 1455, 1280, 1534, 1326, 1613, 1596, 1590, 1575, 1633, 1622, 1561, 1615, 1643, 1574, 1475, 1423, 1607, 1450, 1500, 1641, 1602, 1538, 1649, 1672, 1595, 1603, 1671, 1588, 1636, 1599, 1542, 1564, 1600, 1697, 1716, 1660, 1632, 1632, 1656, 1664, 1604, 1484, 1670, 1704, 1647, 1584, 1528, 1625, 1625, 1636, 1621, 1575, 1625, 1600, 1544, 1578, 1612, 1604, 1672, 1692, 1639, 1669, 1624, 1626, 1627, 1615, 1611, 1608, 1681, 1702, 1689, 1605, 1541, 1570, 1733, 2048, 1592, 1629, 1582, 1616, 1605, 1622, 1612, 1596, 1674, 1693, 1720, 1657, 1706, 1527, 1552, 1584, 1610, 1675, 1693, 1669, 1623, 1569, 1715, 1689, 1629, 1554, 1623, 1691, 1677, 1683, 1615, 1657, 1591, 1482, 1531, 1589, 1690, 1568, 1527, 1449, 1577, 1555, 1444, 1540, 1536, 1611, 1684, 1711, 1667, 1646, 1621, 1742, 1583, 1634, 1623, 1613, 1662, 1560, 1632, 1543, 1645, 1529, 1535, 1603, 1595, 1615, 1641, 1556, 1512, 1685, 1629, 1612, 1593, 1612, 1576, 1666, 1604, 1629, 1575, 1602, 1614, 1516, 1569, 1587, 1608, 162

In [27]:
# chunk 평균 글자수, 최소 글자수, 최대 글자수
import numpy as np
print(np.array([len(document.page_content) for document in documents]).mean())
print(max([len(document.page_content) for document in documents]))
print(min([len(document.page_content) for document in documents]))

1611.5555555555557
2048
1280


## 2.2 1500글짜씩 겹쳐 쪼개 읽어오기

In [39]:
import time
start = time.time()
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("tax_docx/소득세법(법률)(제21221호)(20260701).docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,   # 문서를 쪼갤 때 1500글자씩 chunking
    chunk_overlap=20, # 20문자씩 overlap 하여 chunking
    # separators=["\n\n", "\n", " ", ""]
)
# Recursive(재귀적으로) 다음의 순서대로 chunking 시도 : "\n\n", "\n", " ", ""
documents = loader.load_and_split(text_splitter=text_splitter)
runtime = time.time() - start
print('문서를 1500글자로 겹쳐 쪼개면서 읽는 시간 :', runtime)
print('chunk갯수 :', len(documents))

문서를 1500글자로 겹쳐 쪼개면서 읽는 시간 : 4.104917049407959
chunk갯수 : 180


In [47]:
# chunk들의 글자수
print([len(document.page_content) for document in documents])

[1446, 1421, 1444, 1486, 1188, 1330, 1456, 1257, 1472, 1316, 1347, 1447, 1458, 1343, 1485, 1333, 1478, 1439, 1419, 1445, 1496, 1470, 1455, 1483, 1487, 1382, 1207, 1467, 1399, 1484, 1327, 1420, 1383, 1431, 1387, 1002, 1495, 1484, 1473, 934, 1460, 1471, 1348, 1320, 1285, 1460, 1387, 1478, 1498, 1352, 1418, 1432, 1372, 986, 1441, 1495, 1465, 1439, 1363, 1433, 1475, 1459, 1371, 1398, 1477, 1318, 1417, 1495, 1463, 1443, 1472, 1276, 1391, 1283, 1456, 1354, 1426, 1226, 1492, 1489, 1295, 1483, 1439, 1304, 1451, 1490, 1318, 1364, 1479, 931, 1401, 1478, 1308, 1369, 1491, 1468, 1453, 1414, 1489, 1360, 1458, 1427, 1491, 1495, 1478, 1391, 1416, 1412, 1373, 1349, 1445, 1485, 1367, 1384, 1383, 1452, 1453, 1376, 1419, 1457, 1444, 1419, 1457, 1312, 1467, 1477, 1419, 1475, 1331, 1492, 1491, 1461, 1486, 1479, 1475, 1385, 1305, 1486, 1290, 1492, 1412, 1387, 1495, 1250, 1154, 1330, 1310, 1385, 1462, 1455, 1449, 1365, 1494, 1360, 1426, 1422, 1452, 1388, 1343, 1325, 1467, 1455, 1399, 1375, 1469, 1459, 1138, 

In [43]:
documents[2].page_content

'제2조의3(신탁재산 귀속 소득에 대한 납세의무의 범위) ① 신탁재산에 귀속되는 소득은 그 신탁의 이익을 받을 수익자(수익자가 사망하는 경우에는 그 상속인)에게 귀속되는 것으로 본다.\n\n② 제1항에도 불구하고 위탁자가 신탁재산을 실질적으로 통제하는 등 대통령령으로 정하는 요건을 충족하는 신탁의 경우에는 그 신탁재산에 귀속되는 소득은 위탁자에게 귀속되는 것으로 본다.<개정 2023. 12. 31.>\n\n[본조신설 2020. 12. 29.]\n\n\n\n제3조(과세소득의 범위) ① 거주자에게는 이 법에서 규정하는 모든 소득에 대해서 과세한다. 다만, 해당 과세기간 종료일 10년 전부터 국내에 주소나 거소를 둔 기간의 합계가 5년 이하인 외국인 거주자에게는 과세대상 소득 중 국외에서 발생한 소득의 경우 국내에서 지급되거나 국내로 송금된 소득에 대해서만 과세한다.\n\n② 비거주자에게는 제119조에 따른 국내원천소득에 대해서만과세한다.\n\n③ 제1항 및 제2항을 적용하는 경우 「조세특례제한법」 제100조의14제2호의 동업자에게는 같은 법 제100조의18제1항에 따라 배분받은 소득 및 같은 법 제100조의22제1항에 따라 분배받은 자산의 시가 중 분배일의 지분가액을 초과하여 발생하는 소득에 대하여 과세한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제4조(소득의 구분) ① 거주자의 소득은 다음 각 호와 같이 구분한다. <개정 2013. 1. 1., 2020. 12. 29., 2024. 12. 31.>\n\n1. 종합소득\n\n\u3000이 법에 따라 과세되는 모든 소득에서 제2호 및 제3호에 따른 소득을 제외한 소득으로서 다음 각 목의 소득을 합산한 것\n\n가. 이자소득\n\n나. 배당소득\n\n다. 사업소득\n\n라. 근로소득\n\n마. 연금소득\n\n바. 기타소득\n\n2. 퇴직소득\n\n2의2. 삭제<2024. 12. 31.>\n\n3. 양도소득\n\n② 제1항에 따른 소득을 구분할 때 다음 각 호의 신탁을 제외한 신탁의 이익은 「신탁법」 

# 3. 쪼갠 문서를 임베딩 -> 벡터 데이터 베이스(파일)에 저장
- 임베딩 모델 : text-embedding-3-large (기본모델:text-embedding-ada-002)
- vector DB 저장 폴더 : ./chroma

In [56]:
# 임베딩 객체
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()
embeding = OpenAIEmbeddings(
           # api_key=os.getenv('OPENAI_API_KEY'),
           model = 'text-embedding-3-large'
            )

In [ ]:
# embed_query() : 한 문자열을 임베딩 벡터로 전환
len(embeding.embed_query("소"))

In [58]:
# embed_documents() : 여러 문자열을 임베딩 벡터로 전환
vectors = embeding.embed_documents([
    documents[0].page_content, documents[1].page_content
])

In [60]:
len(vectors[0]), len(vectors[1])

(3072, 3072)